In [2]:
#TODO make sure this renders in the github repo

# Training an LLM

🌟 **LLM training occurs in two phases:**
1. **Pre-Training** (Phase 1):
   - Objective:  Next-Token prediction over a massive, unstructured dataset (trillions of words from the internet, books, code).
   - Goal: To teach the model the structure of human language, logic, coding syntax, and general world knowledge. This is the base model.
2. **Post-Training** (Phase 2):
   - Once the base model is trained, it is undergoes further training so that it can be a useful assistant/chat model. This phase typically involves two sub-steps:
    1. **Supervised Fine-Tuning** (Sub-step 1): The model is trained on thousands of highly curated prompt-and-response pairs. This teaches the mode the format of an assistant. It learns that when it sees questions, is should respond with answers.
    2. **Alignment** (Sub-step 2): Reinforcement Learning from Human Feedback or Direct Preference Optimization is used to score the model's responses. This teaches the model to reject harmful prompts, adhere to safety guidelines, and favour helpful, polite responses.
    - This results in a **Instruct model** (e.g., Llama 3 8B-Instruct)

**Llama 3 paper:**
- **Pre-training:**
  - "We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction. In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”. To do this effectively, pre-training is performed at massive scale: we pre-train a model with 405B parameters on 15.6T tokens using a context window of 8K tokens. This standard pre-training stage is followed by a continued pre-training stage that increases the supported context window to 128K tokens. See Section 3 for details." 
  - "During pre-training on the final 40M tokens, we linearly annealed the learning rate to 0, maintaining a context length of 128K tokens. During this annealing phase, we also adjusted the data mix to upsample data sources of very high quality; see Section 3.1.3. Finally, we compute the average of model checkpoints (Polyak (1991) averaging) during annealing to produce the final pre-trained model."


- **Post-training:**
  - "Similarly, **we adopt a relatively simple post-training procedure based on supervised finetuning (SFT), rejection sampling (RS), and direct preference optimization** (DPO; Rafailov et al. (2023)) as opposed to more complex reinforcement learning algorithms (Ouyang et al., 2022; Schulman et al., 2017) that tend to be less stable and harder to scale."

**Notes:**
- Instead of training on a **number of epochs**, the model is trained on a **number of steps**. Once it reaches the **total_training_steps** (e.g., 1,200,000), training ends. This is done because the datasets for LLM training are so massive, that the model will rarely see the same piece of text twice. Training per epoch just means that for every epoch, the model sees the same dataset.
- The dataloader yields dense, packed batches of tokens.

In [3]:
import EasyJupyter
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import os # TODO change all os to Pathlib

In [4]:
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    from llama_configs import BaseConfig
    from model.decoder import Decoder

## Learning Rate Schedule

**Llama 3 Pre-training:** "We pre-train **Llama 3 405B** using **AdamW** with a **peak learning rate of** $\mathbf{8 × 10^{−5}}$ , **a linear warm up of 8,000 steps**, and **a cosine learning rate schedule decaying to** $\mathbf{8 × 10^{−7}}$ **over** $\mathbf{1{,}200{,}000}$ **steps**. We use a lower batch size early in training to improve training stability, and increase it subsequently to improve efficiency. Specifically, we use an initial batch size of 4M tokens and sequences of length $4{,}096$, and double these values to a batch size of 8M sequences of $8{,}192$ tokens after pre-training 252M tokens. We double the batch size again to 16M after pre-training on $2.87T$ tokens. We found this training recipe to be very stable: we observed few loss spikes and did not require interventions to correct for model training divergence." 

**Llama 2 paper:**
- Hyperparameters. We trained using the **AdamW** optimizer (Loshchilov and Hutter, 2017), with $\boldsymbol{\beta_1 = 0.9}$, $\boldsymbol{\beta_2 = 0.95}$, $\cancel{eps = 10^{−5}}$. ~~We use a cosine learning rate schedule~~, ~~with warmup of~~ $\cancel{2000}$ ~~steps~~, and **decay final learning rate down to 10% of the peak learning rate**. We use a **weight decay of** $\boldsymbol{0.1}$ and ~~gradient clipping of 1.0~~. Figure 5 (a) shows the training loss for Llama 2 with these hyperparameters.

**Note:** In the Llama 3 paper, specifically when they are pre-training the encoders for the **multi-modal Vision** they mentioned "We use a global batch size of $16{,}384$ and a cosine learning rate schedule with initial learning rate $10 × 10^{−4}$ and a weight decay of $0.01$. The initial learning rate was determined based on small-scale experiments."


In [5]:
from torch.optim.lr_scheduler import LambdaLR
import math
import torch.optim as optim

In [6]:
def get_lr_multiplier(
    step: int, warmup_steps: int, total_steps: int, min_lr_ratio: float = 0.01
) -> float:
    """
    Calculates the learning rate multiplier for the Llama 3 schedule.
    1. Linear warmup from 0 to peak_lr.
    2. Linear decay from peak_lr down to 1% of peak_lr.
    """
    # 1. Linear warmup Phase
    if step < warmup_steps:
        # return a fraction from near 0.0 up to 1.0
        return float(step) / float(max(1, warmup_steps))

    # Cosine Decay Phase
    # Cap the step so the learning rate doest drop below the minimum after total steps.
    step = min(step, total_steps)

    # Calculate progress strictly for the decay phase (0.0 to 1.0)
    decay_progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)

    # Cosine: scales down from 1.0 to 0.0
    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * decay_progress))

    # Scale it to the stop at min_lr_ratio (1%) instead of hitting 0
    return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay

In [7]:
def get_std_opt(cfg, model: Decoder):
    """
    Get the Llama 3 scheduler and optimizer.
    """

    optimizer = optim.AdamW(
        model.parameters(),
        lr=cfg.peak_lr,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=0.1,  # TODO verify params in paper
    )

    lr_scheduler = LambdaLR(
        optimizer=optimizer,
        lr_lambda=lambda step: get_lr_multiplier(
            step=step,
            warmup_steps=cfg.warmup_steps,
            total_steps=cfg.total_training_steps,
            min_lr_ratio=0.01
        ),
    )

    return optimizer, lr_scheduler

In [8]:
# @i-c
# Test: LR Schedule Example
print("\n------ LR Schedule Example For The Llama 3 405B Model ------\n")
# Dummy Hyperparameters

total_steps = 1_200_000
warmup_steps = 8_000
peak_lr = 8e-5
min_lr_ratio = 0.01  # 1%


print(f"Total training steps: {total_steps:,}")
print(f"Warmup ends at step: {warmup_steps}")
print(
    f"Peak learning rate: {peak_lr:.8f} | Min learning rate: {peak_lr * min_lr_ratio:.8f}"
)
print(f"\n\n{'Step':>10} | {'Learning-Rate':>15} | {'Phase'}")
print("-" * 55)

milestones = [  # Training milestones
    1,          # Start warmup
    4_000,      # Halfway through warmup
    8_000,      # Peak LR (end of warmup)
    8_001,      # Start of decay (The learning rate immediately starts to drop)
    100_000,    # Early decay
    306_000,    # 25% through decay
    604_000,    # 50% through decay
    902_000,    # 75% through decay
    1_200_000,  # End of decay, hits 1% of peak learning rate
    1_250_000,  # Past total steps (stays flat at 1%)
]

for step in milestones:
    # Get multiplier (0.0 to 1.0)
    multiplier = get_lr_multiplier(
        step, warmup_steps, total_steps, min_lr_ratio=min_lr_ratio
    )

    # Apply it to the peak learning rate
    lr = peak_lr * multiplier

    if step < warmup_steps:
        phase = "Warmup (Linear up ↑)"
    elif step == warmup_steps:
        phase = "Peak LR"
    elif step < total_steps:
        phase = "Decay (Cosine down ↓)"
    else:
        phase = "Minimum LR (Flat →)"

    print(f"{step:10d} | {lr:15.8f} | {phase}")


------ LR Schedule Example For The Llama 3 405B Model ------

Total training steps: 1,200,000
Warmup ends at step: 8000
Peak learning rate: 0.00008000 | Min learning rate: 0.00000080


      Step |   Learning-Rate | Phase
-------------------------------------------------------
         1 |      0.00000001 | Warmup (Linear up ↑)
      4000 |      0.00004000 | Warmup (Linear up ↑)
      8000 |      0.00008000 | Peak LR
      8001 |      0.00008000 | Decay (Cosine down ↓)
    100000 |      0.00007884 | Decay (Cosine down ↓)
    306000 |      0.00006840 | Decay (Cosine down ↓)
    604000 |      0.00004040 | Decay (Cosine down ↓)
    902000 |      0.00001240 | Decay (Cosine down ↓)
   1200000 |      0.00000080 | Minimum LR (Flat →)
   1250000 |      0.00000080 | Minimum LR (Flat →)


## Loss Function

**Llama 3 Paper:** 
- "Language model **pre-training**. **We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction.** **In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”.** To do this effectively, pre-training is performed at massive scale: we pre-train a model with 405B parameters on 15.6T tokens using a context window of 8K tokens. This standard pre-training stage is followed by a continued pre-training stage that increases the supported context window to 128K tokens. See Section 3 for details."
- "Language model **post-training**. The pre-trained language model has a rich understanding of language but it does not yet follow instructions or behave in the way we would expect an assistant to. We align the model with human feedback in several rounds, each of which involves **supervised finetuning (SFT)** on instruction tuning data and **Direct Preference Optimization** (DPO; Rafailov et al., 2024). At this post-training stage, we also integrate new capabilities, such as tool-use, and observe strong improvements in other areas, such as coding and reasoning. See Section 4 for details. Finally, safety mitigations are also incorporated into the model at the post-training stage, the details of which are described in Section 5.4."

**Notes:**
- **Cross-Entropy Loss** was used for the **Pre-Training** (to calculate the error of next-token prediction compared to the ground truth) and in the SFT portion of **Post-Training**. **DPO Loss** is used in the alignment/Direct Preference Optimization portion of **Post-Training**.

## Train Model

In [9]:
import time
from model.utils.model_io import save_checkpoint
from model.utils.plot import plot_loss_history
from model.utils.masking import create_causal_doc_mask
from typing import TYPE_CHECKING

if TYPE_CHECKING:
    from torch.utils.data import DataLoader
    from model.model_text_only import TextOnlyModel


class PreTrainingModel:
    def __init__(self, cfg: BaseConfig, model: TextOnlyModel):
        """
        Implement the pre-training.
        """
        self.cfg = cfg
        self.model = model

        self.criterion = nn.CrossEntropyLoss(
            # label_smoothing=0.1,  # Used Torch's built-in Label Smoothing.
            ignore_index=cfg.special_tokens["pad_token"]["ID"],
            # reduction="sum",
        )
        self.optimizer, self.scheduler = get_std_opt(cfg, model=model)
        # TODO: add loss

        # Track the training steps. Once it reach `total_training_steps` training ends.
        self.step_counter = 0
        self.loss_history = []
        self.start_time = None  # Track the start time of the training

    def train(self, train_dataloader: DataLoader, continue_from_step=0, overfit=False):
        """
        Pre-train the model.

        Args:
            continue_from_step: If continuing training, start from the last step the model was trained on.
            train_dataloader: A dataloader that yields packed batches of tokens.
            overfit: Whether to do a small overfit test.
        """

        cfg = self.cfg
        self.step_counter = continue_from_step

        # Convert the dataloader stream into an iterator for step-based fetching
        data_iter = iter(train_dataloader)

        if overfit:
            print("\n\nOverfitting the model on a single batch......")
            static_x, static_y = next(data_iter)
            grad_accum_steps = 1  # Temporarily override gradient accumulation to 1 for direct updates, needed for overfitting test
        else:
            grad_accum_steps = cfg.gradient_accumulation_steps
            print("\n" + "#" * 64)
            print(
                f"\nPre-Training Model | Total Training Steps: {cfg.total_training_steps:,}"
                f"Accumulation Steps: {grad_accum_steps}"
            )
            print("\n" + "#" * 64)

        self.start_time = time.time()


        while self.step_counter < cfg.total_training_steps:
            # NOTE: If you are training on massive GPU clusters, use HuggingFace's Accelerator for gradient accumulation!
            accumulated_loss = 0.0

            # === Inner loop: Accumulate gradients
            for micro_step in range(grad_accum_steps):
                if overfit:
                    x_ids, y_ids = static_x, static_y
                else:
                    try:
                        # Fetch exactly one micro-batch from the huggingFace stream
                        x_ids, y_ids = next(data_iter)
                    except StopIteration:
                        # Fallback in case a finite dataset exhausts
                        data_iter = iter(train_dataloader)
                        x_ids, y_ids = next(data_iter)

                x_ids = x_ids.to(cfg.device)
                y_ids = y_ids.to(cfg.device)

                # Run micro-batch, and do not call optimizer.step() there!
                loss = self._run_micro_batch(x_ids, y_ids, grad_accum_steps)
                accumulated_loss += loss

            # === Outer loop: The global step
            # Clip gradients to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            # Apply the accumulated gradients to the weights
            self.optimizer.step()
            if not overfit:
                self.scheduler.step()
            self.optimizer.zero_grad()

            self.loss_history.append(accumulated_loss)
            self.step_counter += 1

            if self.step_counter % 50 == 0:
                print(
                    f"Step [{self.step_counter}/{cfg.total_training_steps}] Loss: {accumulated_loss:.4f}"
                )
        cfg.training_stage = "pre-trained"
        save_checkpoint(
            cfg=self.cfg,
            model=self.model,
            optimizer=self.optimizer,
            scheduler=self.scheduler,
            step_counter=self.step_counter,
            loss_history=self.loss_history,
            avg_loss=accumulated_loss,
        )
        plot_loss_history(cfg=cfg, loss_history=self.loss_history)
        print("Training complete!\n\n")

    def _run_micro_batch(self, x_ids, y_ids, grad_accum_steps):
        """
        Run a single forward and backward pass for a micro-batch.

        Args:
            x_ids: The input token ids.
            y_ids: The output token ids.
        """
        # Set the model to train mode
        self.model.train()

        # Generated the doc and causal mask
        batch_mask = create_causal_doc_mask(cfg=self.cfg, token_ids=x_ids)

        # Forward pass
        logits = self.model(x_ids, batch_mask)

        # Compute the loss
        loss = self.criterion(
            logits.view(-1, self.cfg.vocab_size), y_ids.view(-1)
        )  # Flatten to 2D and 1D for CrossEntropy

        # Scale the loss to account for gradient accumulation
        scaled_loss = loss / grad_accum_steps

        # Backward pass
        scaled_loss.backward()

        # Return the unscaled loss
        return loss.item()

## OVERFIT TEST:

In [10]:
# @i-c
from llama_configs import Llama3_scaled_down
from model.model_text_only import TextOnlyModel
from model.tokenizer import  BPETokenizer
from model.utils.data_loader import create_causal_dataloader
from model.utils.data_loader import get_raw_dataset

cfg = Llama3_scaled_down()

# Temporarily override gradient accumulation to 1 for direct updates, needed for overfitting test
cfg.gradient_accumulation_steps = 1
cfg.token_budget = 250 * cfg.global_batch_size_tokens

cfg.model_name = f"{cfg.model_name}_overfit"

tokenizer = BPETokenizer(cfg)

success, trained_tokenizer = tokenizer.load_tokenizer()
if not success:
    trained_tokenizer = tokenizer.train_tokenizer(
        get_raw_dataset(), chunk_size=1000, max_document=1
    )

model = TextOnlyModel(cfg).to(cfg.device)
trainer = PreTrainingModel(cfg, model=model)

/Users/tonyavis/miniconda3/envs/how_to_create_llm/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Project Root: /Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM

Loading existing trained BPE Tokenizer from: (/Users/tonyavis/Main/AI_projects_and_res/How_to_build_an_LLM/model/saved_models/universal_tokenizer/custom_BPE_tokenizer_trained_vocab_size_32000.json)...


Model initialized with 99,528,704 parameters!




KeyboardInterrupt: 

In [ ]:
# @i-c:
# Fetch a single batch
print("\n\nFetching a single batch...")
dataloader = create_causal_dataloader(cfg, tokenizer)

In [ ]:
# @i-c: 
trainer.train(dataloader, overfit=True)

print("If the loss approaches 0.0, the model architecture and gradient flow are functional.")